# Day 2 — LeafGuard dataloader check

Run this on **Kaggle** with the `emmarex/plantdisease` dataset added (Add Data -> search `plantdisease`).
CPU is enough; turn **Internet on** if `leafguard` isn't already uploaded as a Kaggle dataset.

Goal: confirm `leafguard.data.get_dataloaders()` works on the real PlantVillage folder by pulling **one tiny batch**
from each split and printing shapes and labels. Also check the Day 2 fix: train and val splits now
use separate `ImageFolder` instances, so validation gets eval transforms and training keeps its augmentation.
(Since Day 3 the loaders are a seeded 70/15/15 train/val/test split; test uses eval transforms like val.)

In [ ]:
import os

# Kaggle has used both mount layouts; Day 1 found the second one.
CANDIDATE_ROOTS = [
    "/kaggle/input/plantdisease/PlantVillage",
    "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage",
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.isdir(p)), None)
assert DATA_ROOT, f"PlantVillage not found in any of {CANDIDATE_ROOTS} -- is the dataset added?"
print("DATA_ROOT:", DATA_ROOT)

In [ ]:
import subprocess
import sys

# Use an uploaded copy of src/ if there is one; otherwise clone the repo (needs Internet on).
LEAFGUARD_REF = "main"
CLONE_DIR = "/kaggle/working/leafguard"

for path in ["/kaggle/input/leafguard-src", f"{CLONE_DIR}/src"]:
    if os.path.isdir(path) and path not in sys.path:
        sys.path.insert(0, path)

try:
    import leafguard  # noqa: F401
except ImportError:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", LEAFGUARD_REF,
         "https://github.com/Bloop31/leafguard.git", CLONE_DIR],
        check=True,
    )
    sys.path.insert(0, f"{CLONE_DIR}/src")
    import leafguard  # noqa: F401

from leafguard.data import get_dataloaders

print("leafguard", leafguard.__version__, "from", leafguard.__file__)

In [ ]:
import torch

BATCH_SIZE = 8

train_loader, val_loader, test_loader, classes = get_dataloaders(DATA_ROOT, batch_size=BATCH_SIZE, num_workers=2)
train_ds, val_ds, test_ds = train_loader.dataset, val_loader.dataset, test_loader.dataset

print(f"{len(classes)} classes")
print(f"train: {len(train_ds)} images, {len(train_loader)} batches")
print(f"val:   {len(val_ds)} images, {len(val_loader)} batches")
print(f"test:  {len(test_ds)} images, {len(test_loader)} batches")

In [ ]:
def describe_batch(name, loader):
    imgs, labels = next(iter(loader))
    print(f"--- {name} batch")
    print("images:", tuple(imgs.shape), imgs.dtype, f"min={imgs.min():.2f} max={imgs.max():.2f}")
    print("labels:", tuple(labels.shape), labels.dtype, labels.tolist())
    for idx in labels.tolist():
        print(f"  {idx:>2} -> {classes[idx]}")
    return imgs, labels


train_imgs, train_labels = describe_batch("train", train_loader)
val_imgs, val_labels = describe_batch("val", val_loader)

assert train_imgs.shape == (BATCH_SIZE, 3, 224, 224)
assert val_imgs.shape == (BATCH_SIZE, 3, 224, 224)
assert int(train_labels.max()) < len(classes) and int(val_labels.max()) < len(classes)

In [ ]:
# Day 2 fix: train wraps its own ImageFolder, so its transforms are independent of val/test.
train_tfms = train_ds.dataset.transform
val_tfms = val_ds.dataset.transform
print("train transforms:", [type(t).__name__ for t in train_tfms.transforms])
print("val transforms:  ", [type(t).__name__ for t in val_tfms.transforms])

assert train_ds.dataset is not val_ds.dataset
assert any("Random" in type(t).__name__ for t in train_tfms.transforms), "train lost its augmentation"
assert not any("Random" in type(t).__name__ for t in val_tfms.transforms), "val is still augmented"
assert test_ds.dataset.transform is val_tfms, "test should use the same eval transforms as val"
train_idx, val_idx, test_idx = set(train_ds.indices), set(val_ds.indices), set(test_ds.indices)
assert train_idx.isdisjoint(val_idx) and train_idx.isdisjoint(test_idx) and val_idx.isdisjoint(test_idx), "splits overlap"
assert len(train_ds) + len(val_ds) + len(test_ds) == len(train_ds.dataset)

# Same image twice: val output is identical, train output differs (augmentation is live).
assert torch.equal(val_ds[0][0], val_ds[0][0])
print("train sample differs between draws:", not torch.equal(train_ds[0][0], train_ds[0][0]))
print("All dataloader checks passed.")

## Findings (fill this in after running)

- Number of classes / train size / val size / test size:
- Batch shapes and label dtype:
- Did the transform check pass?
- Anything unexpected (slow loading, corrupt images, wrong class names)?